# Foundation 07 — Context and Evidence Security

Compile authorized, current, provenance-bound evidence into a bounded manifest; surface insufficiency and conflicts; and verify drafted claims against exact admitted references before release.

![Context compiler and evidence release boundary](architecture.svg)

Trusted request context and a current registry govern admission, authority, lifecycle, conflict, and budget. The model sees evidence as data; deterministic release checks bind claims and citations to the manifest.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
import sys
sys.path.insert(0, '.')
ns = runpy.run_path('lab.py')
CompilationStatus, ReleaseStatus = ns['CompilationStatus'], ns['ReleaseStatus']
CandidateRef, DraftClaim = ns['CandidateRef'], ns['DraftClaim']
ContextCompiler, ReleaseVerifier = ns['ContextCompiler'], ns['ReleaseVerifier']
build_registry, request_for, NOW = ns['build_registry'], ns['request_for'], ns['NOW']
registry = build_registry()
request = request_for('retention_days','request:notebook')

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
policy = registry.records['policy:retention']
manifest = ContextCompiler(registry).compile(request,(CandidateRef.exact(policy),),now=NOW)
draft = (DraftClaim('retention_days','30',(policy.reference,)),)
release = ReleaseVerifier(registry).verify(manifest,draft,now=NOW)
assert manifest.status is CompilationStatus.READY and release.status is ReleaseStatus.ANSWERED
{'manifest':manifest.manifest_digest,'resolved':manifest.resolved_claims,'release':release}

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
south = registry.records['case:south:9']
cross = ContextCompiler(registry).compile(request_for('case_status','request:cross'),(CandidateRef.exact(south),),now=NOW)
assert cross.status is CompilationStatus.INSUFFICIENT
assert cross.decisions[0].reason == 'tenant' and not cross.entries
cross

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
conflict_refs = tuple(CandidateRef.exact(registry.records[item]) for item in ('case:conflict-a','case:conflict-b'))
conflict = ContextCompiler(registry).compile(request_for('case_owner','request:conflict'),conflict_refs,now=NOW)
launder_manifest = ContextCompiler(registry).compile(request,(CandidateRef.exact(policy),CandidateRef.exact(registry.records['case:42'])),now=NOW)
launder = ReleaseVerifier(registry).verify(launder_manifest,(DraftClaim('retention_days','30',(registry.records['case:42'].reference,)),),now=NOW)
assert conflict.status is CompilationStatus.CONFLICT
assert launder.status is ReleaseStatus.BLOCKED and launder.reason == 'citation-does-not-support-claim'
(conflict,launder)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
report,cases = ns['evaluate_controls']()
assert (report.cases,report.valid_cases,report.attack_cases,report.failure_cases) == (14,4,8,2)
assert report.valid_task_success_rate == report.attack_block_rate == 1
assert report.attack_effect_rate == report.unauthorized_admission_rate == report.stale_admission_rate == 0
{'populations':{'all':report.cases,'valid':report.valid_cases,'attack':report.attack_cases,'failure':report.failure_cases},'rates':report,'outcomes':{case.name:(case.manifest.status.value,case.release.status.value,case.release.reason) for case in cases}}

## 6. Exercise a second failure mode

In [ ]:
policy_error = ContextCompiler(registry,policy_available=False).compile(request,(CandidateRef.exact(policy),),now=NOW)
unavailable = build_registry(); unavailable.available = False
registry_error = ContextCompiler(unavailable).compile(request,(CandidateRef.exact(unavailable.records['policy:retention']),),now=NOW)
assert policy_error.status is registry_error.status is CompilationStatus.ERROR
(policy_error,registry_error)

## 7. Authority is not similarity or recency

A lower-authority observation may be authentic and newer but cannot override current policy. The compiler records both while resolving the required claim from the configured authority class.

In [ ]:
refs = tuple(CandidateRef.exact(registry.records[item]) for item in ('observation:retention','policy:retention'))
authority_manifest = ContextCompiler(registry).compile(request,refs,now=NOW)
resolved = authority_manifest.resolved_claims[0]
assert resolved.value == '30' and resolved.authority.name == 'POLICY'
{'resolved':resolved,'entries':[(entry.source_id,entry.authority.name) for entry in authority_manifest.entries]}

## 8. Budget exclusion cannot downgrade evidence

When the only authoritative support does not fit the declared budget, the compiler recomputes sufficiency and abstains instead of answering from a cheaper record.

In [ ]:
costly = ns['make_evidence']('policy:costly',1,'north',ns['SourceKind'].POLICY,ns['Classification'].CONFIDENTIAL,ns['Authority'].POLICY,'policy://costly',(('retention_days','30'),),'Thirty days.',NOW-timedelta(days=1),NOW+timedelta(days=1),cost_units=5)
budget_registry = ns['EvidenceRegistry']({costly.source_id:costly})
budgeted = ContextCompiler(budget_registry).compile(request,(CandidateRef.exact(costly),),now=NOW,max_context_units=4)
assert budgeted.status is CompilationStatus.INSUFFICIENT and budgeted.decisions[0].reason == 'budget-excluded'
budgeted

## 9. OpenAI Agents SDK: manifest-scoped read tool

The real strict function tool accepts only an exact evidence reference. Identity, tenant, clearance, purpose, policy, and manifest binding stay in trusted runtime context; no model or network call is made.

In [ ]:
sdk = runpy.run_path('sdk_adapter.py')
sdk_release,evidence = sdk['credential_free_demo']()
assert sdk_release.status.value == 'answered' and evidence['strict_json_schema']
assert evidence['schema_fields'] == ['evidence_ref'] and evidence['trusted_fields_excluded']
{'release':sdk_release,'tool':evidence['tool_name'],'schema_fields':evidence['schema_fields']}

## 10. Production replacement

Production replacement: authenticated identity; claim-specific policy; durable evidence catalog with owner, schema, tenant and subject scope, purpose, classification, lifecycle, effective time, expiry, revocation, supersession, digest or signature, and derivation lineage; security trimming before ranking; isolated caches; deterministic context budgets; immutable run manifests; structured claim and citation output; current reference and support verification; explicit abstention, block, deny, and error semantics; protected privacy-aware receipts; and continuous safety, utility, conflict, freshness, citation, chaos, and adversarial evaluation. The local dictionaries, ordinal authority, timestamps, and hashes prove only an in-process control sequence.

## 11. Exercises

1. Make authority claim-specific.
2. Add a low-risk freshness grace period and prohibit it for policy.
3. Detect false corroboration through shared root lineage.
4. Rotate a source between compile and release.
5. Map the manifest to W3C PROV or OpenLineage.
6. Add an authorized vector-store adapter while preserving admission-before-ranking.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.